# Qwen3.5-4B: 15 questions


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers==5.22.1", "accelerate==1.15.0"],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==5.22.1', 'accelerate==1.15.0'], returncode=0)

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
import os
from pathlib import Path
import shutil
import time
import traceback
import zipfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch
from transformers import (
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
    Qwen3_5ForConditionalGeneration,
    set_seed,
)

run_name = os.environ.get("PLANT_RUN_NAME", "qwen35_4b_nonthinking")
dataset_path = Path("/content/data/evaluation/init_questions/questions.jsonl")
run_dir = Path("/content/data/runs/init_questions") / run_name
config = {
    "model_id": "Qwen/Qwen3.5-4B",
    "model_revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    "dataset_name": "init_questions",
    "enable_thinking": False,
    "dtype": "float16",
    "attention_implementation": "sdpa",
    "seed": 42,
    "seed_policy": "seed + int(question_id)",
    "max_input_tokens": 1024,
    "generation": {
        "max_new_tokens": 512,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "repetition_penalty": 1.0,
    },
    "presence_penalty": 1.5,
    "system_prompt": (
        "Ты помощник по уходу за комнатными растениями, садом и огородом. "
        "Отвечай новичку на русском языке кратко и понятно, объясняй причины советов. "
        "Если совет зависит от недостающих условий, задавай уточняющие вопросы. "
        "Обозначай неопределённость и не выдумывай сведения."
    ),
}
print(run_name)

qwen35_4b_nonthinking


In [ ]:
if not dataset_path.exists():
    from google.colab import files

    uploaded = files.upload()
    if "questions.jsonl" not in uploaded:
        raise FileNotFoundError("Upload questions.jsonl from data/evaluation/init_questions/.")
    dataset_path.parent.mkdir(parents=True, exist_ok=True)
    dataset_path.write_bytes(uploaded["questions.jsonl"])

dataset_bytes = dataset_path.read_bytes()
questions = [json.loads(line) for line in dataset_bytes.decode().splitlines() if line.strip()]
if [row["id"] for row in questions] != [f"{i:02}" for i in range(1, 16)]:
    raise ValueError("Expected questions 01-15 in their original order.")
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This notebook requires an NVIDIA T4.")

run_dir.mkdir(parents=True, exist_ok=True)
config_path = run_dir / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != config:
    raise ValueError("This run already has different settings. Choose a new run name.")
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
shutil.copy2(dataset_path, run_dir / "questions.jsonl")

environment = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "cuda": torch.version.cuda,
    "packages": {name: version(name) for name in [
        "torch", "transformers", "accelerate", "tokenizers", "safetensors", "huggingface-hub"
    ]},
    "nvidia_smi": subprocess.check_output(["nvidia-smi"], text=True),
}
(run_dir / "environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2) + "\n"
)
(run_dir / "requirements-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)
print(json.dumps(environment, ensure_ascii=False, indent=2))

{
  "started_at_utc": "2026-10-31T09:09:45.109247+00:00",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "gpu": "Tesla T4",
  "gpu_memory_gib": 14.56317138671875,
  "compute_capability": [
    7,
    5
  ],
  "cuda": "13.0",
  "packages": {
    "torch": "2.11.0+cu130",
    "transformers": "5.22.1",
    "accelerate": "1.15.0",
    "tokenizers": "0.23.3",
    "safetensors": "0.8.0",
    "huggingface-hub": "1.33.0"
  },
  "nvidia_smi": "Sat Oct 31 09:09:45 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |\n|                                         |

In [ ]:
load_started = time.perf_counter()
torch.cuda.reset_peak_memory_stats()
tokenizer = AutoTokenizer.from_pretrained(
    config["model_id"], revision=config["model_revision"], token=False
)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    config["model_id"],
    revision=config["model_revision"],
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation=config["attention_implementation"],
    use_safetensors=True,
    token=False,
)
model.eval()
load_stats = {
    "load_seconds": time.perf_counter() - load_started,
    "allocated_gib": torch.cuda.memory_allocated() / 2**30,
    "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
    "model_class": type(model).__name__,
}
(run_dir / "model-loading.json").write_text(json.dumps(load_stats, indent=2) + "\n")
print(json.dumps(load_stats, indent=2))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


{
  "load_seconds": 129.53349087499998,
  "allocated_gib": 8.455068588256836,
  "peak_reserved_gib": 8.46484375,
  "model_class": "Qwen3_5ForConditionalGeneration"
}


In [ ]:
class PresencePenalty(LogitsProcessor):
    def __init__(self, penalty, prompt_length):
        self.penalty = penalty
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated = input_ids[:, self.prompt_length:]
        if generated.numel():
            seen = torch.zeros_like(scores, dtype=torch.bool)
            seen.scatter_(1, generated, True)
            scores = scores - seen.to(scores.dtype) * self.penalty
        return scores


answers_path = run_dir / "answers.jsonl"
answers = [json.loads(line) for line in answers_path.read_text().splitlines()] if answers_path.exists() else []
if len({row["id"] for row in answers}) != len(answers):
    raise ValueError("Duplicate question IDs in saved answers.")


def save_artifacts():
    ordered = sorted(answers, key=lambda row: row["id"])
    summary = {
        "run_name": run_name,
        "completed": len(ordered),
        "expected": len(questions),
        "output_tokens": sum(row["output_tokens"] for row in ordered),
        "generation_seconds": sum(row["elapsed_seconds"] for row in ordered),
        "truncated_ids": [row["id"] for row in ordered if row["finish_reason"] == "length"],
        "thinking_marker_ids": [row["id"] for row in ordered if row["thinking_markers"]],
        "peak_reserved_gib": max([load_stats["peak_reserved_gib"]] + [row["peak_reserved_gib"] for row in ordered]),
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    sections = ["# Qwen3.5-4B answers", f"Run: `{run_name}`. Non-thinking mode."]
    for row in ordered:
        sections.extend([f"## Question {row['id']}", row["question"], row["answer"]])
    (run_dir / "answers.md").write_text("\n\n".join(sections) + "\n")
    with zipfile.ZipFile(run_dir / "artifacts.zip", "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run_dir.iterdir()):
            if path.is_file() and path.name != "artifacts.zip":
                archive.write(path, arcname=path.name)


def answer_question(row):
    if row["id"] in {answer["id"] for answer in answers}:
        return
    seed = config["seed"] + int(row["id"])
    set_seed(seed)
    messages = [
        {"role": "system", "content": config["system_prompt"]},
        {"role": "user", "content": row["question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda:0")
    input_length = inputs["input_ids"].shape[1]
    if input_length > config["max_input_tokens"]:
        raise ValueError(f"Question {row['id']} exceeds the input token limit.")
    penalty = PresencePenalty(config["presence_penalty"], input_length)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            **config["generation"],
            logits_processor=LogitsProcessorList([penalty]),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated = output[0, input_length:]
    raw_response = tokenizer.decode(generated, skip_special_tokens=False)
    eos_ids = model.generation_config.eos_token_id
    eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]
    finished = bool(generated.numel() and generated[-1].item() in eos_ids)
    answer = {
        "id": row["id"],
        "question": row["question"],
        "messages": messages,
        "rendered_prompt": prompt,
        "answer": tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "raw_response": raw_response,
        "generation_seed": seed,
        "input_tokens": input_length,
        "output_tokens": generated.numel(),
        "elapsed_seconds": elapsed,
        "tokens_per_second": generated.numel() / elapsed,
        "finish_reason": "eos" if finished else "length",
        "thinking_markers": "<think>" in raw_response or "</think>" in raw_response,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
    }
    with answers_path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(answer, ensure_ascii=False) + "\n")
    answers.append(answer)
    save_artifacts()
    print(f"{row['id']}/15: {answer['output_tokens']} tokens, {elapsed:.1f} s, {answer['finish_reason']}", flush=True)
    print(answer["answer"], flush=True)

In [ ]:
try:
    answer_question(questions[0])
except Exception:
    (run_dir / "error.txt").write_text(traceback.format_exc())
    save_artifacts()
    raise

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


01/15: 482 tokens, 35.4 s, eos


Размножение эпипремнума (суккулентной лианы) стеблевыми черенками — отличный способ, так как растение быстро укореняется. Вот пошаговая инструкция:

1.  **Подготовка черенка**
    *   Возьмите здоровый стебель длиной около 10–15 см.
    *   **Важно:** Срезайте его под углом, чтобы площадь соприкосновения с грунтом была больше.
    *   Оставьте на срезе немного «хвостика» листа вверху (это поможет быстрее начать расти), но не отрезайте листья полностью сразу, если хотите получить несколько побегов.

2.  **Сушка среза**
    *   Это критический этап для суккулентов! Чтобы не гнило, оставьте черенок открытым на воздухе на 1–3 дня.
    *   *Почему:* Срез должен подсохнуть и образовать защитную корку. Если посадить мокрый черенок прямо в землю, он начнет гнить из-за избыточной влаги внутри тканей растения.

3.  **Выбор места посадки**
    *   Подготовьте горшок с дренажом и легким грунтом (можно смесь универсальной с песком).
    *   Поставьте черенок в горшок так, чтобы он стоял вертикально

In [ ]:
try:
    for row in questions[1:]:
        answer_question(row)
except Exception:
    (run_dir / "error.txt").write_text(traceback.format_exc())
    raise
finally:
    save_artifacts()

print((run_dir / "summary.json").read_text())
print(f"Results archive: {run_dir / 'artifacts.zip'}")

02/15: 291 tokens, 19.0 s, eos


Частоту полива орхидеи нельзя назвать одним конкретным числом, так как она зависит от сезона и условий содержания.

**Общее правило:** Поливайте орхидею **только когда субстрат (корневая среда) просохнет**.
*   **Причина:** Орхидеи не любят застоя воды в корнях. Если поливать слишком часто, корни гниют даже при комнатной температуре.
*   **Как проверить:** Вдавите палец в субстрат на глубину 2–3 см. Поливайте только если он feels сухим. Если чувствуете влагу — ждите еще пару дней.

**Уточняющие вопросы для точного совета:**
Чтобы дать более точный график, уточните:
1.  Какой у вас тип орхидеи (например, *Фаленопсис*, *Цимбидиум* или дикорастущая)?
2.  Где именно находится растение (на балконе, в ванной или в сухом помещении с кондиционером)?
3.  Какую среду вы используете для посадки (мох сфагнум, кора, керамзит или грунт из магазина)?

Без этих данных сложно сказать, сколько раз в неделю нужно поливать именно вашу конкретную орхидею.


03/15: 207 tokens, 13.7 s, eos


Как искусственный интеллект, я не могу дать вам ответ на основе личных эстетических предпочтений или субъективного мнения. Красота — понятие индивидуальное: для кого-то это яркое лиственное растение (например, монстера), а для кого-то — изящный цветок с мелкими лепестками (например, фиалка).

Поскольку вы просили назвать **одно** объективно красивее всех остальных, такой выбор невозможен без нарушения принципа объективности и выдумывания фактов.

Однако я могу помочь вам выбрать идеальное растение, если вы расскажете немного больше о ваших условиях:
*   Где именно оно будет стоять (светлый подоконник, тень, офис)?
*   Сколько времени в день освещается место?
*   Есть ли возможность поливать его регулярно?

Скажите эти детали, и я предложу конкретный вариант, который будет выглядеть наилучшим образом при данных условиях.


04/15: 271 tokens, 17.9 s, eos


Закаливание нужно, чтобы помочь рассаде постепенно привыкнуть к условиям улицы и не получить ожоги от солнца или стресс от перепадов температур.

**Как это сделать:**
1.  **Начните с вечера.** Выводите растения на балкон или во двор (под навес) за 30–40 минут до заката. Они будут в тени.
2.  **Увеличивайте время.** Каждые 2–3 дня оставляйте их на улице всё дольше: сначала на 1 час, потом на 3–4 часа, пока не сможете держать их на улице весь световой день.
3.  **Дневная защита.** Если на улице пасмурно, можно выносить растения сразу утром и вечером. Но если стоит яркое солнце, лучше оставлять их под тенью или заносить обратно к вечеру, чтобы избежать ожогов листьев.

**Важное уточнение:**
Чтобы дать точный совет о сроках и безопасности, мне нужно знать:
*   На какой улице вы живете (какая там температура)?
*   Как выглядит ваша рассада (высокая/низкая, сколько ей месяцев)?
*   Есть ли у вас возможность ставить её под тень или только на открытый балкон?


... (19 outputs left out)

Такого сорта **не существует**.

Причина: урожайность томата напрямую зависит от условий (свет, температура, полив, почва) и ухода. То, что дает хороший урожай в одном климате, может не плодоносить или погибнуть в другом без коррекции ухода. Даже самые неприхотливые сорта требуют базового внимания.

**Что я могу сделать:**
Назову сорт, который считается одним из самых **устойчивых и простых в выращивании новичкам** (например, «Сливовый» или «Черри»), но уточните условия вашего региона и способа выращивания (на грядке или на балконе), чтобы я дал конкретный совет по уходу для него.


15/15: 177 tokens, 11.7 s, eos


Для подкормки рассады перца в **1 литре воды** обычно добавляют **2–3 мл** жидкого удобрения.

**Причина:** Рассада находится на стадии активного роста, но ей еще не нужно много азота (чтобы не вытянуться), поэтому концентрацию следует держать умеренной. Стандартная инструкция для большинства жидких удобрений рекомендует разводить их в соотношении 1:500 или 1:1000, что и дает этот объем.

**Важное уточнение:**
Поскольку я не знаю марку удобрения (концентрации могут отличаться даже у разных производителей), пожалуйста, проверьте этикетку на банке. Там должно быть написано точное количество миллилитров на 1 литр воды. Если там указано иное, следуйте именно этой цифре.


{
  "run_name": "qwen35_4b_nonthinking",
  "completed": 15,
  "expected": 15,
  "output_tokens": 4317,
  "generation_seconds": 288.9102007069994,
  "truncated_ids": [],
  "thinking_marker_ids": [],
  "peak_reserved_gib": 8.615234375
}

Results archive: /content/data/runs/init_questions/qwen35_4b_nonthinking/artifacts.zip
